In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets huggingface_hub torch tqdm pandas scipy requests matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Create directories for results
import os
os.makedirs('results', exist_ok=True)

# Required Libraries

In [ ]:
import json
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import time
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import gc

# Download Dataset

In [ ]:
def download_task_data(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to download data: {response.status_code}")

In [ ]:
# URL of the task.json file
task_url = "https://raw.githubusercontent.com/google/BIG-bench/main/bigbench/benchmark_tasks/logical_deduction/five_objects/task.json"

In [ ]:
print("Downloading logical deductions task data...")
task_data = download_task_data(task_url)

## Basic Dataset Information

In [ ]:
print(f"Dataset name: {task_data['name']}")
print(f"Description: {task_data['description']}")
print(f"Number of examples: {len(task_data['examples'])}")

Dataset name: five_objects
Description: A five-object logical deduction task which requires deducing the order of a sequence of objects
Number of examples: 500


## Task Prefix

In [ ]:
task_prefix = task_data.get('task_prefix', '')
print(f"Task prefix: {task_prefix}")

Task prefix: The following paragraphs each describe a set of five objects arranged in a fixed order. The statements are logically consistent within each paragraph.




## First Example

In [ ]:
print("\nExample input:")
print(task_data['examples'][0]['input'])
print("\nExample target scores:")
for option, score in task_data['examples'][0]['target_scores'].items():
    print(f"  {option}: {score}")


Example input:
On a shelf, there are five books: a gray book, a red book, a purple book, a blue book, and a black book. The red book is to the right of the gray book. The black book is to the left of the blue book. The blue book is to the left of the gray book. The purple book is the second from the right.

Example target scores:
  The gray book is the leftmost.: 0
  The red book is the leftmost.: 0
  The purple book is the leftmost.: 0
  The blue book is the leftmost.: 0
  The black book is the leftmost.: 1


## Subset of Data

In [ ]:
max_samples = min(200, len(task_data['examples']))
examples = task_data['examples'][:max_samples]
print(f"\nUsing {len(examples)} examples for benchmarking")


Using 200 examples for benchmarking


# Mixtral8x7b Quantized 4bit

In [ ]:
# Authenticate with HuggingFace
print("Please enter your HuggingFace token when prompted:")
login()

Please enter your HuggingFace token when prompted:


## Load Model

In [ ]:
try:
    print("Loading Mixtral tokenizer...")
    model_id = "mistralai/Mixtral-8x7B-Instruct-v0.1"
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    print("Loading Mixtral model with 4-bit quantization...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        load_in_4bit=True,
        torch_dtype=torch.bfloat16,
    )
    print("Model loaded successfully")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

Loading Mixtral tokenizer...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

model-00018-of-00019.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00019-of-00019.safetensors:   0%|          | 0.00/4.22G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully


# Utility Functions

In [ ]:
def format_logical_deduction_prompt(example):
    """
    Format a logical deduction task example into a prompt for the Mixtral model.
    """
    # Extract input and options
    input_text = example['input']
    options = list(example['target_scores'].keys())

    # Start with the task prefix
    formatted_prompt = task_prefix + input_text + "\n\nChoose the correct answer from the following options:\n"

    # Add multiple choice options
    for i, option in enumerate(options):
        letter = chr(65 + i)  # A, B, C, D, E
        formatted_prompt += f"{letter}) {option}\n"

    # Add instruction to only output the letter of the answer
    formatted_prompt += "\nPlease answer with just the letter of the correct option (A, B, C, D, or E)."

    return formatted_prompt, options

In [ ]:
def generate_response(prompt, max_new_tokens=20, max_retries=3):
    """
    Generate a response from the Mixtral model using the chat template.
    Ensures only a concise answer is given.
    """
    # Format according to Mixtral's chat template
    formatted_prompt = f"<s>[INST] {prompt} [/INST]"

    # Add retry logic
    for attempt in range(max_retries):
        try:
            inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=0.1,  # Low temperature for deterministic outputs
                    do_sample=False,   # Greedy decoding
                    pad_token_id=tokenizer.eos_token_id
                )

            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            response = response[len(tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)):].strip()

            return response
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"Generation attempt {attempt+1} failed: {e}. Retrying...")
                # Clean memory before retry
                gc.collect()
                torch.cuda.empty_cache()
            else:
                print(f"All generation attempts failed. Last error: {e}")
                return "ERROR: Generation failed"

In [ ]:
def extract_answer_choice(response, num_options=5):
    """
    Extract the answer choice (A, B, C, D, E) from the model's response.
    Returns the letter or None if no valid answer is found.
    """
    response = response.strip().upper()

    # Valid options (adjust based on the number of choices)
    valid_options = [chr(65 + i) for i in range(num_options)]  # A, B, C, D, E

    # First check if the response is simply a valid option letter
    for option in valid_options:
        if option == response or response.startswith(option + ".") or response.startswith(option + ")"):
            return option

    # Check if the response contains a clear option marker
    for option in valid_options:
        if f"OPTION {option}" in response or f"ANSWER {option}" in response or f"ANSWER: {option}" in response:
            return option

    # Look for the first valid option letter in the response
    for char in response:
        if char in valid_options:
            return char

    # If no clear answer choice, return None
    return None

In [ ]:
def get_correct_option(example):
    """
    Get the correct option (letter) from the example's target scores.
    """
    options = list(example['target_scores'].keys())
    scores = list(example['target_scores'].values())
    correct_index = scores.index(1)  # Assume score of 1 indicates the correct answer
    return chr(65 + correct_index)  # Convert to letter (A, B, C, D, E)

# Testing with Smaller Sample

In [ ]:
# Test with a few examples
print("Testing with a small sample...")
test_size = 3
test_results = []

for i, example in enumerate(examples[:test_size]):
    # Format prompt and get options
    prompt, options = format_logical_deduction_prompt(example)
    print(f"\nTest {i+1}:")
    print(f"Prompt: {prompt}")

    # Generate response
    response = generate_response(prompt)
    print(f"Response: {response}")

    # Extract answer choice
    answer_choice = extract_answer_choice(response, len(options))
    print(f"Extracted answer: {answer_choice}")

    # Get correct answer
    correct_answer = get_correct_option(example)
    print(f"Correct answer: {correct_answer}")

    # Check if the answer is correct
    is_correct = answer_choice == correct_answer if answer_choice else False
    print(f"Correct: {is_correct}")

    # Add to test results
    test_results.append({
        'example_id': i,
        'prompt': prompt,
        'response': response,
        'extracted_answer': answer_choice,
        'correct_answer': correct_answer,
        'is_correct': is_correct,
        'is_ambiguous': answer_choice is None
    })

# Convert to DataFrame and check results
test_df = pd.DataFrame(test_results)
print("\nTest results:")
print(test_df[['example_id', 'extracted_answer', 'correct_answer', 'is_correct', 'is_ambiguous']])

# Check if the response extraction is working correctly
if test_df['is_ambiguous'].any():
    print("\nWARNING: Some responses couldn't be parsed into a clear answer choice.")
    print("You may need to adjust the extract_answer_choice function or the prompt.")

Testing with a small sample...

Test 1:
Prompt: The following paragraphs each describe a set of five objects arranged in a fixed order. The statements are logically consistent within each paragraph.

On a shelf, there are five books: a gray book, a red book, a purple book, a blue book, and a black book. The red book is to the right of the gray book. The black book is to the left of the blue book. The blue book is to the left of the gray book. The purple book is the second from the right.

Choose the correct answer from the following options:
A) The gray book is the leftmost.
B) The red book is the leftmost.
C) The purple book is the leftmost.
D) The blue book is the leftmost.
E) The black book is the leftmost.

Please answer with just the letter of the correct option (A, B, C, D, or E).


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Response: A) The gray book is the leftmost.

Here's the reasoning:
1
Extracted answer: A
Correct answer: E
Correct: False

Test 2:
Prompt: The following paragraphs each describe a set of five objects arranged in a fixed order. The statements are logically consistent within each paragraph.

On a shelf, there are five books: a gray book, a red book, a purple book, a blue book, and a black book. The red book is to the right of the gray book. The black book is to the left of the blue book. The blue book is to the left of the gray book. The purple book is the second from the right.

Choose the correct answer from the following options:
A) The gray book is the second from the left.
B) The red book is the second from the left.
C) The purple book is the second from the left.
D) The blue book is the second from the left.
E) The black book is the second from the left.

Please answer with just the letter of the correct option (A, B, C, D, or E).
Response: C) The purple book is the second from the

# Main Evaluation

In [ ]:
def evaluate_logical_deduction(examples, save_every=10, save_prefix="logical_deduction"):
    """
    Evaluate the Mixtral model on logical deduction examples.

    Args:
        examples: List of task examples
        save_every: Save intermediate results every n samples
        save_prefix: Prefix for saved files

    Returns:
        DataFrame containing the evaluation results
    """
    results = []
    errors = 0

    start_time = time.time()

    for idx, example in enumerate(tqdm(examples)):
        try:
            # Format prompt and get options
            prompt, options = format_logical_deduction_prompt(example)

            # Generate response
            response = generate_response(prompt)

            # Skip if response generation failed
            if response.startswith("ERROR"):
                print(f"Skipping example {idx} due to generation error")
                errors += 1
                continue

            # Extract answer choice
            answer_choice = extract_answer_choice(response, len(options))

            # Get correct answer
            correct_answer = get_correct_option(example)

            # Check if the answer is correct
            is_correct = answer_choice == correct_answer if answer_choice else False

            # Store result
            result = {
                'example_id': idx,
                'input_text': example['input'],
                'options': options,
                'prompt': prompt,
                'model_response': response,
                'extracted_answer': answer_choice,
                'correct_answer': correct_answer,
                'is_correct': is_correct,
                'is_ambiguous': answer_choice is None
            }

            results.append(result)

            # Save intermediate results
            if (idx + 1) % save_every == 0:
                save_path = f'results/{save_prefix}_results_intermediate_{idx+1}.csv'
                pd.DataFrame(results).to_csv(save_path, index=False)

                elapsed_time = time.time() - start_time
                avg_time_per_sample = elapsed_time / (idx + 1)
                estimated_total_time = avg_time_per_sample * len(examples)
                remaining_time = estimated_total_time - elapsed_time

                print(f"Processed {idx+1}/{len(examples)} examples")
                print(f"Elapsed time: {elapsed_time/60:.2f} minutes")
                print(f"Estimated time remaining: {remaining_time/60:.2f} minutes")
                print(f"Errors so far: {errors}")

                # Clean memory periodically
                gc.collect()
                torch.cuda.empty_cache()

        except Exception as e:
            errors += 1
            print(f"Error processing example {idx}: {e}")
            # Continue with the next example
            continue

    if errors > 0:
        print(f"Completed with {errors} errors out of {len(examples)} examples.")

    return pd.DataFrame(results)

## Run Full Evaluation

In [ ]:
# Run the full evaluation
print("\nRunning full evaluation...")
start_time = time.time()

results_df = evaluate_logical_deduction(examples, save_every=10)

# Calculate total time
total_time = time.time() - start_time
print(f"Total evaluation time: {total_time/60:.2f} minutes")

# Save the full results
results_df.to_csv('results/logical_deduction_full_results.csv', index=False)


Running full evaluation...


  0%|          | 0/200 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Processed 10/200 examples
Elapsed time: 0.46 minutes
Estimated time remaining: 8.71 minutes
Errors so far: 0
Processed 20/200 examples
Elapsed time: 0.94 minutes
Estimated time remaining: 8.45 minutes
Errors so far: 0
Processed 30/200 examples
Elapsed time: 1.35 minutes
Estimated time remaining: 7.66 minutes
Errors so far: 0
Processed 40/200 examples
Elapsed time: 1.83 minutes
Estimated time remaining: 7.32 minutes
Errors so far: 0
Processed 50/200 examples
Elapsed time: 2.29 minutes
Estimated time remaining: 6.86 minutes
Errors so far: 0
Processed 60/200 examples
Elapsed time: 2.78 minutes
Estimated time remaining: 6.49 minutes
Errors so far: 0
Processed 70/200 examples
Elapsed time: 3.26 minutes
Estimated time remaining: 6.05 minutes
Errors so far: 0
Processed 80/200 examples
Elapsed time: 3.68 minutes
Estimated time remaining: 5.51 minutes
Errors so far: 0
Processed 90/200 examples
Elapsed time: 4.13 minutes
Estimated time remaining: 5.05 minutes
Errors so far: 0
Processed 100/200 e

# Calculate Performance Metrics

In [ ]:
# Calculate overall metrics
total_examples = len(results_df)
correct_examples = results_df['is_correct'].sum()
incorrect_examples = (~results_df['is_correct'] & ~results_df['is_ambiguous']).sum()
ambiguous_examples = results_df['is_ambiguous'].sum()

accuracy = correct_examples / total_examples
error_rate = incorrect_examples / total_examples
ambiguity_rate = ambiguous_examples / total_examples

print(f"\n===== OVERALL EVALUATION METRICS =====")
print(f"Model: Mixtral-8x7B-Instruct-v0.1 (4-bit quantized)")
print(f"Dataset: BigBench Logical Deduction (Five Objects)")
print(f"Number of examples: {total_examples}")
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Error Rate: {error_rate:.4f}")
print(f"Ambiguity Rate: {ambiguity_rate:.4f}")

# Calculate performance by answer position
answer_position_metrics = results_df.groupby('correct_answer').agg({
    'is_correct': 'mean',
    'example_id': 'count'
}).rename(columns={'is_correct': 'accuracy', 'example_id': 'count'})

print("\n===== PERFORMANCE BY CORRECT ANSWER POSITION =====")
print(answer_position_metrics)

# Calculate metrics for selected vs. correct answers
# This shows if the model has any position biases
print("\n===== SELECTED VS. CORRECT ANSWER ANALYSIS =====")
selection_matrix = pd.crosstab(
    results_df['extracted_answer'],
    results_df['correct_answer'],
    normalize='columns'  # Normalize by correct answer
)
print(selection_matrix)


===== OVERALL EVALUATION METRICS =====
Model: Mixtral-8x7B-Instruct-v0.1 (4-bit quantized)
Dataset: BigBench Logical Deduction (Five Objects)
Number of examples: 200

Accuracy: 0.4150
Error Rate: 0.5850
Ambiguity Rate: 0.0000

===== PERFORMANCE BY CORRECT ANSWER POSITION =====
                accuracy  count
correct_answer                 
A                  0.525     40
B                  0.325     40
C                  0.350     40
D                  0.325     40
E                  0.550     40

===== SELECTED VS. CORRECT ANSWER ANALYSIS =====
correct_answer        A      B      C      D      E
extracted_answer                                   
A                 0.525  0.150  0.025  0.175  0.150
B                 0.025  0.325  0.025  0.000  0.050
C                 0.050  0.025  0.350  0.050  0.075
D                 0.075  0.175  0.200  0.325  0.175
E                 0.325  0.325  0.400  0.450  0.550


# Visualizations

In [ ]:
# Set the style
plt.style.use('ggplot')
sns.set(font_scale=1.2)

# Create a directory for visualizations
os.makedirs('results/visualizations', exist_ok=True)

# 1. Overall Accuracy Pie Chart
fig1, ax1 = plt.subplots(figsize=(10, 8))
labels = ['Correct', 'Incorrect', 'Ambiguous']
sizes = [correct_examples, incorrect_examples, ambiguous_examples]
colors = ['#5cb85c', '#d9534f', '#f0ad4e']

ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title('Overall Results', fontsize=16)
plt.savefig('results/visualizations/overall_results_pie.png', dpi=300)
plt.close()

# 2. Accuracy by Answer Position
fig2, ax2 = plt.subplots(figsize=(12, 8))
sns.barplot(x=answer_position_metrics.index, y='accuracy', data=answer_position_metrics, ax=ax2)
ax2.set_title('Accuracy by Correct Answer Position', fontsize=16)
ax2.set_xlabel('Correct Answer')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1)

# Add count labels
for i, (idx, row) in enumerate(answer_position_metrics.iterrows()):
    ax2.text(i, row['accuracy'] + 0.02, f"n={int(row['count'])}", ha='center')
    ax2.text(i, row['accuracy'] - 0.05, f"{row['accuracy']:.2f}", ha='center', color='white', fontweight='bold')

plt.savefig('results/visualizations/accuracy_by_position.png', dpi=300)
plt.close()

# 3. Confusion Matrix - Selected vs. Correct Answers
fig3, ax3 = plt.subplots(figsize=(12, 10))
sns.heatmap(selection_matrix, annot=True, fmt='.2f', cmap='Blues', ax=ax3)
ax3.set_title('Confusion Matrix: Selected vs. Correct Answers', fontsize=16)
ax3.set_xlabel('Correct Answer')
ax3.set_ylabel('Selected Answer')
plt.savefig('results/visualizations/confusion_matrix.png', dpi=300)
plt.close()

# 4. Model's Answer Distribution
fig4, ax4 = plt.subplots(figsize=(12, 8))
answer_counts = results_df['extracted_answer'].value_counts().sort_index()
answer_counts = answer_counts.reindex(sorted(answer_counts.index.fillna('None')))

sns.barplot(x=answer_counts.index, y=answer_counts.values, ax=ax4)
ax4.set_title('Model\'s Answer Distribution', fontsize=16)
ax4.set_xlabel('Selected Answer')
ax4.set_ylabel('Count')

# Add percentage labels
for i, count in enumerate(answer_counts.values):
    percentage = count / total_examples * 100
    ax4.text(i, count + 5, f"{percentage:.1f}%", ha='center')

plt.savefig('results/visualizations/answer_distribution.png', dpi=300)
plt.close()

# 5. Learning Curve - Accuracy over Time
fig5, ax5 = plt.subplots(figsize=(14, 8))
# Calculate rolling accuracy
window_size = min(20, len(results_df) // 10)
rolling_acc = results_df['is_correct'].rolling(window=window_size).mean()

ax5.plot(range(len(results_df)), rolling_acc, 'b-')
ax5.set_title(f'Rolling Accuracy (Window Size: {window_size})', fontsize=16)
ax5.set_xlabel('Example Index')
ax5.set_ylabel('Accuracy')
ax5.set_ylim(0, 1)
ax5.grid(True)
plt.savefig('results/visualizations/learning_curve.png', dpi=300)
plt.close()

print("\nVisualizations saved to results/visualizations/ directory")


Visualizations saved to results/visualizations/ directory
